# Taller resuelto: DBSCAN

## Objetivo
Aplicar DBSCAN para detectar clusters no lineales y puntos de ruido, manteniendo la estructura de explicación antes, código comentado e interpretación después.

## Bloque 1. Importar librerías

Importaremos herramientas para generar datos no lineales, escalar, aplicar DBSCAN, evaluar y comparar.

In [ ]:
# Importamos NumPy para operaciones numéricas.
import numpy as np

# Importamos Matplotlib para graficar.
import matplotlib.pyplot as plt

# Importamos make_moons para crear datos no lineales.
from sklearn.datasets import make_moons

# Importamos StandardScaler para escalar datos.
from sklearn.preprocessing import StandardScaler

# Importamos DBSCAN.
from sklearn.cluster import DBSCAN

# Importamos silhouette_score para evaluación.
from sklearn.metrics import silhouette_score


### Interpretación

DBSCAN trabaja con densidad. No necesita número de clusters, pero sí depende de `eps` y `min_samples`.

## Bloque 2. Generar datos no lineales

Crearemos datos con forma de medias lunas para mostrar un caso donde los clusters no son esféricos.

In [ ]:
# Generamos datos con forma de medias lunas.
X, y_true = make_moons(n_samples=500, noise=0.07, random_state=42)

# Escalamos los datos para que las distancias sean comparables.
X_scaled = StandardScaler().fit_transform(X)

# Mostramos la forma de los datos.
print("Shape de X:", X_scaled.shape)

# Creamos una figura.
plt.figure(figsize=(7,5))

# Dibujamos las clases reales solo como referencia.
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=y_true, cmap="viridis", edgecolors="k")

# Agregamos título y etiquetas.
plt.title("Dataset no lineal make_moons")
plt.xlabel("Variable 1 escalada")
plt.ylabel("Variable 2 escalada")

# Mostramos la gráfica.
plt.show()


### Interpretación

La estructura curva muestra por qué un algoritmo basado en centroides puede fallar. DBSCAN es adecuado porque sigue regiones densas.

## Bloque 3. Aplicar DBSCAN

Entrenaremos DBSCAN con parámetros iniciales y revisaremos los clusters detectados.

In [ ]:
# Creamos el modelo DBSCAN.
dbscan = DBSCAN(eps=0.3, min_samples=5)

# Entrenamos el modelo y obtenemos etiquetas.
labels = dbscan.fit_predict(X_scaled)

# Mostramos etiquetas únicas.
print("Etiquetas encontradas:", np.unique(labels))

# Contamos puntos marcados como ruido.
print("Puntos de ruido:", np.sum(labels == -1))


### Interpretación

La etiqueta `-1` representa ruido. Esta es una ventaja de DBSCAN: puede detectar puntos que no pertenecen claramente a ningún cluster.

## Bloque 4. Visualizar DBSCAN

Graficaremos el resultado para ver si DBSCAN respeta la forma no lineal del dataset.

In [ ]:
# Creamos figura.
plt.figure(figsize=(7,5))

# Dibujamos puntos con color según etiqueta DBSCAN.
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=labels, cmap="viridis", edgecolors="k")

# Agregamos título y etiquetas.
plt.title("Clusters encontrados por DBSCAN")
plt.xlabel("Variable 1 escalada")
plt.ylabel("Variable 2 escalada")

# Mostramos la gráfica.
plt.show()


### Interpretación

Si los colores siguen las dos lunas, DBSCAN capturó correctamente la geometría de los datos. Los puntos de ruido aparecen diferenciados por su etiqueta.

## Bloque 5. Evaluar con Silhouette excluyendo ruido

Calcularemos Silhouette solo sobre puntos asignados a clusters válidos.

In [ ]:
# Creamos una máscara para excluir ruido.
mask = labels != -1

# Verificamos si hay más de un cluster válido.
if len(np.unique(labels[mask])) > 1:
    # Calculamos Silhouette sin ruido.
    score = silhouette_score(X_scaled[mask], labels[mask])
    
    # Mostramos resultado.
    print("Silhouette sin ruido:", score)
else:
    # Mostramos mensaje si no es posible calcular.
    print("No hay suficientes clusters válidos para calcular Silhouette")


### Interpretación

En DBSCAN, la evaluación debe considerar que hay ruido. Por eso no siempre se interpreta igual que en K-Means.

## Bloque 6. Comparar distintos valores de eps

Probamos varios radios de vecindad para observar cómo cambia el agrupamiento.

In [ ]:
# Definimos valores de eps.
eps_values = [0.15, 0.25, 0.35, 0.50]

# Creamos subgráficas.
fig, axes = plt.subplots(1, len(eps_values), figsize=(18,4))

# Recorremos cada valor de eps.
for ax, eps in zip(axes, eps_values):
    # Creamos DBSCAN con eps actual.
    model = DBSCAN(eps=eps, min_samples=5)
    
    # Entrenamos y obtenemos etiquetas.
    pred = model.fit_predict(X_scaled)
    
    # Dibujamos resultado.
    ax.scatter(X_scaled[:,0], X_scaled[:,1], c=pred, cmap="viridis", edgecolors="k")
    
    # Agregamos título.
    ax.set_title(f"eps = {eps}")

# Ajustamos espacios.
plt.tight_layout()

# Mostramos figura.
plt.show()


### Interpretación

`eps` pequeño puede crear mucho ruido. `eps` grande puede fusionar clusters. Este parámetro controla la sensibilidad del algoritmo.

## Bloque 7. Comparar DBSCAN con K-Means

Entrenaremos K-Means sobre los mismos datos para comparar distancia vs densidad.

In [ ]:
# Importamos KMeans.
from sklearn.cluster import KMeans

# Creamos K-Means con 2 clusters.
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)

# Entrenamos y predecimos.
labels_kmeans = kmeans.fit_predict(X_scaled)

# Creamos figura comparativa.
_, axes = plt.subplots(1,2,figsize=(12,5))

# Graficamos K-Means.
axes[0].scatter(X_scaled[:,0], X_scaled[:,1], c=labels_kmeans, cmap="viridis", edgecolors="k")
axes[0].set_title("K-Means")

# Graficamos DBSCAN.
axes[1].scatter(X_scaled[:,0], X_scaled[:,1], c=labels, cmap="viridis", edgecolors="k")
axes[1].set_title("DBSCAN")

# Mostramos figura.
plt.show()


### Interpretación

K-Means tiende a dividir por cercanía a centroides. DBSCAN sigue la densidad y por eso se adapta mejor a formas curvas.

## Cierre

DBSCAN es útil cuando hay ruido y formas complejas, pero requiere cuidado con `eps` y `min_samples`.